# 1 · The keynote catalogue

Every row of `data/catalog/events.csv` is one keynote: when it started (with its IANA time
zone, because the event day depends on whether the market was still open), whether a live
audience was present, and which YouTube video carries the full broadcast audio. The
catalogue was researched company by company and validated by `applause catalog`
(`src/unpacked_applause/catalog.py`); manual decisions are listed in
`data/catalog/curation_log.md`.

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display
from unpacked_applause.config import FIGURES, PROCESSED, TABLES
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [ ]:
import duckdb
from unpacked_applause.catalog import load_events
events = load_events()
con = duckdb.connect()
con.register("events", events.assign(date_local=events["date_local"].dt.date))
con.execute(open("../sql/coverage.sql").read()).df()

## When do keynotes happen relative to the trading day?

`t0` is the first session whose close is after the keynote start. Apple, Google and
Microsoft usually present at 10:00 Pacific, i.e. during US trading hours, so the reaction
starts the same day; Tesla presents in the evening and Samsung's Unpacked is overnight in
Seoul, so their `t0` is the next session.

In [ ]:
study = pd.read_parquet(PROCESSED / "event_study.parquet")
ev = events.merge(study[["event_id", "t0", "days_to_t0"]], on="event_id", how="left")
ev["t0_same_day"] = ev["days_to_t0"] == 0
ev.groupby("company")["t0_same_day"].mean().rename("share with t0 = keynote day").to_frame()

In [ ]:
display(Image(filename=str(FIGURES / "coverage_by_year.png")))

In [ ]:
events.groupby(["company", "audience"]).size().unstack(fill_value=0)